# Pre-Deployment QA — Trading Algorithm Readiness Checklist

Run every cell against a fresh backtest trade log **before** deploying to the cloud.
Each section prints a PASS/FLAG verdict. Treat any FLAG as a blocker to investigate,
not a number to explain away.

Set `LOG_PATH` to the trade log you want to vet.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

LOG_PATH = 'trade_log_ETRADEV2_0.csv'   # <- point at the log under test
WIN = ('WIN_T1_T2','WIN_T1','WIN_T1_BE')

df = pd.read_csv(LOG_PATH)
df['win'] = df['result'].isin(WIN)
df['_day'] = pd.to_datetime(df['day'])
df = df.sort_values('_day').reset_index(drop=True)
print(f'{len(df)} trades | {df["_day"].min().date()} -> {df["_day"].max().date()}')

## 1. Sample size & statistical confidence

The first question is never 'is it profitable' — it's 'do I have enough trades to believe the number.' A win rate from 27 trades is a rumor; from 552 it's evidence. The 95% confidence interval shows how wide the uncertainty still is.

In [ ]:
n = len(df)
wr = df['win'].mean()
se = np.sqrt(wr*(1-wr)/n) if n else 0
lo, hi = wr-1.96*se, wr+1.96*se
tier = ('anecdote (<30)' if n<30 else 'weak (<100)' if n<100 else
        'usable (100-500)' if n<500 else 'solid (500+)')
print(f'trades          : {n}  -> {tier}')
print(f'win rate        : {wr*100:.1f}%')
print(f'95% CI          : [{lo*100:.1f}%, {hi*100:.1f}%]  (width {(hi-lo)*100:.1f} pts)')
print('FLAG: sample too small — do not trust hourly/regime cuts' if n<100 else 'PASS: sample adequate for aggregate stats')

## 2. Expectancy, profit factor & win/loss asymmetry

Profitability decomposed: expectancy is your edge per trade; profit factor is gross-win / gross-loss (>1.5 is healthy, >2 strong); and the avg-win vs avg-loss ratio tells you whether you win big and lose small, or the reverse.

In [ ]:
pnl = df['pnl_dollars']
wins, losses = pnl[pnl>0], pnl[pnl<=0]
exp = pnl.mean()
pf = wins.sum()/abs(losses.sum()) if losses.sum()!=0 else np.inf
print(f'expectancy      : ${exp:.2f} / trade')
print(f'profit factor   : {pf:.2f}')
print(f'avg win / loss  : ${wins.mean():.0f} / ${losses.mean():.0f}  '
      f'(ratio {abs(wins.mean()/losses.mean()):.2f})')
print(f'net P&L         : ${pnl.sum():,.0f}')
print('PASS: positive expectancy & PF>1.5' if exp>0 and pf>1.5 else 'FLAG: weak edge')

## 3. Drawdown & equity curve

Max drawdown is the deepest peak-to-trough dip — the pain you must survive. Compare it to the account size and your prop-firm trailing limit. A smooth up-and-to-the-right curve is trustworthy; a curve carried by two lucky spikes is not.

In [ ]:
eq = pnl.cumsum(); peak = eq.cummax(); dd = eq - peak
maxdd = dd.min()
ACCOUNT = 50000
print(f'max drawdown    : ${maxdd:,.0f}  ({maxdd/ACCOUNT*100:.1f}% of ${ACCOUNT:,})')
print(f'final equity    : ${eq.iloc[-1]:,.0f}')
fig, ax = plt.subplots(1,2, figsize=(12,3.2))
ax[0].plot(eq.values); ax[0].set_title('Equity curve'); ax[0].axhline(0, color='k', lw=.5)
ax[1].fill_between(range(len(dd)), dd.values, 0, color='crimson', alpha=.5); ax[1].set_title('Drawdown ($)')
plt.tight_layout(); plt.show()

## 4. Losing streaks — the psychological & trailing-limit test

Backers and prop-firm trailing limits die to *consecutive* losses, not the average. If your worst historical streak would breach the eval, the strategy isn't deployable at that size even if it's profitable overall.

In [ ]:
is_loss = (~df['win']).astype(int).values
cur=mx=0; streaks=[]
for x in is_loss:
    cur = cur+1 if x else 0; mx=max(mx,cur)
# worst cumulative $ loss during any losing run
run_pnl=0; worst_run=0
for p in pnl:
    run_pnl = min(0, run_pnl+p) if p<0 else 0
    worst_run = min(worst_run, run_pnl)
print(f'worst losing streak      : {mx} trades in a row')
print(f'worst cumulative run loss: ${worst_run:,.0f}')
print('Compare the run loss to your trailing-drawdown limit before sizing.')

## 5. Out-of-sample split — the overfitting test

**The single most important test.** Split the log by *time* (never randomly — that leaks future into past). If the recent 30% performs far worse than the early 70%, your parameters were curve-fit to history and will decay live. Similar performance across both halves is the strongest evidence of a real edge.

In [ ]:
cut = int(len(df)*0.7)
tr, te = df.iloc[:cut], df.iloc[cut:]
def line(name, d):
    print(f'{name:6}: {len(d):>4} trades | ${d["pnl_dollars"].mean():>7.1f}/trade | '
          f'win {d["win"].mean()*100:4.1f}% | PF {d["pnl_dollars"][d["pnl_dollars"]>0].sum()/abs(d["pnl_dollars"][d["pnl_dollars"]<=0].sum()):.2f}')
line('train', tr); line('test', te)
decay = (te['pnl_dollars'].mean()-tr['pnl_dollars'].mean())/abs(tr['pnl_dollars'].mean())*100
print(f'test vs train expectancy: {decay:+.0f}%')
print('FLAG: test collapses — likely overfit' if decay < -40 else 'PASS: edge holds out of sample')

## 6. Data integrity & lookahead detection

Catches the bugs that make a backtest lie: entries logged *before* the zone formed (lookahead — the deadliest), non-positive stops, zero-contract trades, missing timestamps. These invalidate every other number above, so run this first in spirit even though it's listed last.

In [ ]:
issues = []
ft = pd.to_datetime(df['formed_at'], utc=True)
et = pd.to_datetime(df['entry_time'], utc=True)
if (et < ft).any():           issues.append(f'LOOKAHEAD: {(et<ft).sum()} entries before zone formed')
if (df['stop_pts']<=0).any(): issues.append(f'{(df["stop_pts"]<=0).sum()} non-positive stops')
if (df['contracts']<1).any(): issues.append(f'{(df["contracts"]<1).sum()} sub-1 contract trades')
if df['entry_time'].isna().any(): issues.append('missing entry_time')
if df['pnl_dollars'].isna().any(): issues.append('missing pnl')
print('PASS: no integrity issues' if not issues else 'FLAG:\n  - ' + '\n  - '.join(issues))

## 7. Consistency across regimes

A robust edge works in more than one market. Slice by the `trend` / `curve` context your strategy records. If profit comes from a single regime, you're exposed the moment the market leaves it — fine to know, dangerous not to.

In [ ]:
for col in ['trend','curve','zone_type']:
    if col in df.columns:
        g = df.groupby(col).agg(trades=('win','size'), win=('win','mean'),
                                pnl=('pnl_dollars','sum'), avg=('pnl_dollars','mean'))
        g['win'] = (g['win']*100).round(1); g['pnl']=g['pnl'].round(0); g['avg']=g['avg'].round(1)
        print(f'--- by {col} ---'); print(g.to_string()); print()

## Verdict

Green-light for cloud deploy only when: sample is adequate (1), expectancy & PF are positive (2), max drawdown fits your account & limit (3,4), **the out-of-sample half holds up (5)**, integrity is clean (6), and profit isn't from a single regime (7). Any FLAG is a stop, not a footnote.

Adopt this as the gate every strategy version clears before it leaves the notebook.